In [1]:
%load_ext autoreload
%autoreload 2
from sindex.sources.openalex.snapshot import create_oa_pubdate_table
import duckdb

In [2]:
config = {
    "meta_out": r"C:\Users\Admin\Documents\OpenAlex\full-snapshot\metadata",
    "db_path": r"I:\pipeline-data\external\openalex-snapshot\duckdb\oa_snapshot.duckdb",
    "temp": r"C:/Users/Admin/Documents/OpenAlex/full-snapshot/duckdb_temp",
}

In [8]:
create_oa_pubdate_table(
    db_path=config["db_path"], 
    meta_folder = config["meta_out"], 
    mem_limit = "32GB", 
    temp_dir = config["temp"],
    file_limit = None,
    reset_tables = True
)

Resetting tables...
Starting ingestion of 1716 files into 'openalex_pubdate'...
Processed: 1716/1716 | Elapsed: 00:32:55
Done! Total items in 'openalex_pubdate': 462,346,925


In [10]:
con = duckdb.connect(config["db_path"])
row_count = con.execute("SELECT count() FROM openalex_pubdate").fetchone()[0]
print(f"Total OpenAlex works in table: {row_count:,}")
display(con.execute("SELECT * FROM openalex_pubdate LIMIT 20").df())
con.close()

Total OpenAlex works in table: 462,346,925


,oa_id,doi,pubdate
0,W869536896,,2004-01-01
1,W654252834,,2007-09-01
2,W3120397303,10.30465/crtls.2020.5285,2020-05-21
3,W3143375225,,2020-01-01
4,W2736735170,,2009-01-01
5,W3021674332,,2016-11-01
6,W323442416,,1972-09-05
7,W587957745,,2009-02-24
8,W3031921790,10.3760/cma.j.issn.1004-6461.2005.11.021,2005-11-25
9,W3092365388,,1982-01-01


<bound method pybind11_detail_function_record_v1_msvc_md_mscver19.close of <_duckdb.DuckDBPyConnection object at 0x00000298DDC6E1B0>>

In [3]:
#Create index on doi
con = duckdb.connect(config["db_path"])
con.execute("CREATE INDEX IF NOT EXISTS idx_doi ON openalex_pubdate (doi);")
print("Index created successfully.")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Index created successfully.


In [6]:
con = duckdb.connect(config["db_path"])
test = con.execute("EXPLAIN SELECT pubdate FROM openalex_pubdate WHERE doi = '10.30465/crtls.2020.5285'").fetchone()[1]
if "INDEX_SCAN" in test:
    print("Success! DuckDB is using the index.")
else:
    print("DuckDB is still doing a full table scan.")
print(test)
con.close()

DuckDB is still doing a full table scan.
┌───────────────────────────┐
│         SEQ_SCAN          │
│    ────────────────────   │
│           Table:          │
│      openalex_pubdate     │
│                           │
│   Type: Sequential Scan   │
│    Projections: pubdate   │
│                           │
│          Filters:         │
│  doi='10.30465/crtls.2020 │
│           .5285'          │
│                           │
│          ~4 rows          │
└───────────────────────────┘



In [ ]:
10.5281/zenodo.12827591

In [8]:
con = duckdb.connect(config["db_path"])
test = con.execute("SELECT pubdate FROM openalex_pubdate WHERE doi = '10.30465/crtls.2020.5285'").fetchone()[0]
print(test)
con.close()

2020-05-21


In [9]:
con = duckdb.connect(config["db_path"])
test = con.execute("SELECT pubdate FROM openalex_pubdate WHERE doi = '10.5281/zenodo.12827591'").fetchone()[0]
print(test)
con.close()

2024-07-25


In [10]:
con = duckdb.connect(config["db_path"])

stats = con.execute("""
    SELECT 
        COUNT(*), 
        COUNT(NULLIF(doi, '')) 
    FROM openalex_pubdate
""").fetchone()

print(f"Total Rows: {stats[0]:,}")
print(f"Non-Empty DOIs: {stats[1]:,}")
print(f"Coverage: {(stats[1]/stats[0])*100:.2f}%")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total Rows: 462,346,925
Non-Empty DOIs: 281,874,059
Coverage: 60.97%
